# Wild Boar (*Sus scrofa*) Resistance Surface Pipeline

**Cantons covered:** Aargau (training) and Zurich (transfer).

**Goal.** Build a 25 m wild boar resistance surface from GPS telemetry,
following Fischer et al. (2024), Clontz et al. (2021), and Keeley et
al. (2016). The model is calibrated on Aargau in-matrix steps and then
transferred to Zurich.

**Pipeline.**

1. Load and clean telemetry.
2. Compute regularised movement metrics.
3. Classify steps as in-patch or in-matrix using a hierarchical rule
   classifier.
4. Prepare spatial covariates from SwissTLM3D, DHM25, barriers, and
   wildlife passages.
5. Fit an integrated step-selection function (iSSF) on in-matrix
   steps.
6. Convert iSSF coefficients to a resistance surface (Keeley
   exponential transformation with C = 4).
7. Transfer the resistance surface to Canton Zurich, reusing the
   Aargau scale parameters and eta range.
8. Diagnostic plots and quick output verification.

**Data sources.**

- Telemetry: `WS_Masterfile_final_with_cultures_Aargau.csv`.
- SwissTLM3D 2026 (LV95): land cover, water, roads, buildings.
- DHM25 (LV03): terrain elevation.
- Wildlife passages: `Wildtierpassagen.gdb`, layer
  `N2023_Version_Wildtierpassagen`.
- Cantonal boundaries: `swissBOUNDARIES3D`.
- Permanent ASP fences (Zurich): `Dauerzaeune.shp` from VetAmt Zurich.

**Note on study area extent.** Every raster is computed on a buffered
rectangle around the canton (2 km buffer, clipped to the Swiss
national border so Germany is excluded). The buffer is required to
keep `focal()`, `distance()`, and bilinear projections free of edge
effects near the canton boundary. The FINAL resistance and suitability
rasters in `data/processed/` are then masked to the exact canton
polygon. Intermediate covariate layers in `data/temp/` keep the
buffered extent for debugging.


## Setup: Packages and Configuration


In [ ]:
if (!require("pacman")) install.packages("pacman")

pacman::p_load(
  # Movement ecology
  amt,        # Step-selection functions, track resampling
  # Spatial
  terra,      # Raster and vector operations
  sf,         # Simple Features vector operations
  # Data wrangling
  dplyr,
  tidyr,
  purrr,
  lubridate,
  # Modelling
  survival,   # Conditional logistic regression (clogit)
  broom,      # Tidy model output
  # Visualisation
  ggplot2
)


In [ ]:
# =============================================================================
# CONFIGURATION (adjust paths before running)
# =============================================================================

# --- Input paths -------------------------------------------------------------
TELEMETRY_PATH  <- "../data/raw/WS_Masterfile_final_with_cultures_Aargau.csv"
TLM_PATH        <- "../data/raw/SWISSTLM3D_2026_LV95_LN02.gpkg"
DEM_PATH        <- "../data/raw/dhm25_grid_raster.asc"
BOUNDARIES_PATH <- "../data/raw/swissBOUNDARIES3D_1_5_LV95_LN02.gpkg"
PASSAGES_GDB    <- "../data/raw/Wildtierpassagen.gdb"
PASSAGES_LAYER  <- "N2023_Version_Wildtierpassagen"
FENCE_PATH      <- "../data/raw/Daten_Bericht_VetAmt/Daten_Bericht_VetAmt/Daten/shapefiles/Dauerzaeune/Dauerzaeune.shp"

# --- Output paths ------------------------------------------------------------
OUT_DIR  <- "../data/processed/"
TEMP_DIR <- "../data/temp/"
dir.create(OUT_DIR,  recursive = TRUE, showWarnings = FALSE)
dir.create(TEMP_DIR, recursive = TRUE, showWarnings = FALSE)

# --- Spatial settings --------------------------------------------------------
TARGET_CRS   <- "EPSG:2056"   # Swiss LV95
DEM_CRS      <- "EPSG:21781"  # LV03, native CRS of DHM25
TARGET_RES_M <- 25            # Raster resolution in metres
BUFFER_M     <- 2000          # Rectangle buffer around each canton

# --- iSSF settings -----------------------------------------------------------
N_RANDOM_STEPS <- 10   # Random control steps per observed step
KEELEY_C       <- 4    # Exponential transformation constant (Fischer 2024)

# --- Telemetry settings ------------------------------------------------------
FIX_RATE_MIN <- 15    # Nominal GPS fix interval in minutes
FIX_TOL_MIN  <- 5     # Tolerance around the fix interval in minutes
MIN_DAYS     <- 30    # Minimum tracking duration per individual
MIN_FIXES    <- 100   # Minimum raw fixes per individual after QC

# Individuals excluded from the analysis. Document any addition here so
# the rationale travels with the code.
#   - 027_Agat_13970 : short re-collaring deployment with poor fix
#                      success (50.6 % on-schedule, 32 % short gaps,
#                      16.5 % medium gaps). The same animal is well
#                      represented by the 027_Agat_13969 deployment
#                      with 14 595 clean fixes.
EXCLUDE_IDS <- c("027_Agat_13970")

# --- Large lake threshold ----------------------------------------------------
# Lakes larger than this become absolute barriers (NA in the final
# resistance raster). Smaller water bodies stay as soft barriers.
LAKE_MIN_AREA_M2 <- 1e6   # 1 km^2 (Hallwilersee, Greifensee scale)

# --- Land cover classifications (SwissTLM3D objektart) ----------------------
FOREST_CLASSES <- c("Wald", "Wald offen", "Gebueschwald", "Gehoelzflaeche")
EXCLUDE_AREALS <- c("Wald nicht bestockt", "Abbauareal", "Obstanlage",
                    "Friedhof", "Reben", "Gewaesserareal")

# --- Road classes ------------------------------------------------------------
FENCED_ROAD_CLASSES    <- c("Autobahn", "Autostrasse", "Ausfahrt", "Einfahrt",
                            "Zufahrt", "Autozug")
MAINROAD_CLASSES       <- c("10m Strasse", "8m Strasse", "6m Strasse")
SECONDARY_ROAD_CLASSES <- c("4m Strasse", "3m Strasse", "2m Weg",
                            "Verbindung", "Dienstzufahrt", "Platz")

# KUNSTBAUTE values that mean the linear feature is NOT at ground
# level (tunnels, underpasses, inside buildings). Used by
# prep_above_ground() in Step 4e to filter such features out. Both
# umlaut and ASCII variants are listed because SwissTLM3D exports
# differ across versions.
KUNSTBAUTE_UNDERGROUND <- c("Tunnel", "Unterfuehrung", "Unterführung",
                            "in Gebaeude", "in Gebäude")

# --- Settlement areas (polygon layer tlm_namen_siedlungsname) ---------------
# These classes describe built-up areas at different scales (town,
# district, neighborhood, sub-neighborhood). Using all four gives
# full coverage.
SETTLEMENT_LAYER   <- "tlm_namen_siedlungsname"
SETTLEMENT_CLASSES <- c("Ort", "Ortsteil", "Quartier", "Quartierteil")

cat("Configuration loaded.\n")


## Step 1: Load and Clean Telemetry

Load the raw CSV, parse timestamps, drop a 5 day capture-shock buffer
per individual, and keep only animals with at least `MIN_DAYS` of
tracking data and `MIN_FIXES` raw fixes. Individuals listed in
`EXCLUDE_IDS` are removed here as well.


In [ ]:
cat("== 1. Loading telemetry ==\n")

raw <- read.csv(TELEMETRY_PATH, sep = ";", stringsAsFactors = FALSE,
                fileEncoding = "UTF-8")

tel <- raw %>%
  mutate(
    datetime_utc = dmy_hms(paste(DatumUTC, ZeitUTC), tz = "UTC"),
    id           = Tier
  ) %>%
  # Spatial and temporal validity, with a plausibility bound for Swiss
  # national coordinates so any obviously bad row is dropped.
  filter(
    !is.na(datetime_utc),
    !is.na(X), !is.na(Y),
    X > 400000, X < 900000,
    Y >  50000, Y < 350000
  ) %>%
  arrange(id, datetime_utc) %>%
  # Drop the first 5 days per animal as capture-shock buffer.
  group_by(id) %>%
  filter(datetime_utc >= (min(datetime_utc) + days(5))) %>%
  ungroup() %>%
  # Drop the pre-defined problem individuals listed in EXCLUDE_IDS.
  filter(!id %in% EXCLUDE_IDS) %>%
  # Compute per-individual deployment metrics and apply the minimum
  # duration and fix-count thresholds.
  group_by(id) %>%
  mutate(
    n_days  = as.numeric(difftime(max(datetime_utc), min(datetime_utc),
                                  units = "days")),
    n_fixes = n()
  ) %>%
  ungroup() %>%
  filter(n_days >= MIN_DAYS, n_fixes >= MIN_FIXES)

data.frame(
  Metric = c("Individuals retained", "Total GPS fixes"),
  Value  = c(n_distinct(tel$id), nrow(tel))
)


## Step 2: Compute Regularised Movement Metrics

Resample each track to the nominal 15 minute fix interval and compute
step lengths `sl_` (metres) and turning angles `ta_` (radians) via
`amt::steps_by_burst()`. Steps are computed per burst, so any gap
longer than `FIX_RATE_MIN + FIX_TOL_MIN` ends a burst and never
produces a spurious long step.


In [ ]:
cat("== 2. Computing movement metrics ==\n")

# Build an amt track in the configured target CRS.
trk <- tel %>%
  make_track(X, Y, datetime_utc, id = id, crs = TARGET_CRS)

# Resample each individual to the nominal fix interval.
# track_resample() keeps at most one fix per (FIX_RATE_MIN +/- FIX_TOL_MIN)
# window, which also collapses the burst-pair duplicates documented in
# the methods (see the resampling-loss diagnostic in the next cell).
trk_resampled <- trk %>%
  nest(data = -id) %>%
  mutate(data = map(data, function(d) {
    tryCatch(
      track_resample(d,
                     rate      = minutes(FIX_RATE_MIN),
                     tolerance = minutes(FIX_TOL_MIN)),
      error = function(e) NULL
    )
  })) %>%
  filter(!map_lgl(data, is.null)) %>%
  unnest(data)

# Convert resampled tracks to steps (sl_ in metres, ta_ in radians).
steps_all <- trk_resampled %>%
  nest(data = -id) %>%
  mutate(steps = map(data, steps_by_burst)) %>%
  select(id, steps) %>%
  unnest(steps)

# Step-length distribution as a quick sanity check.
data.frame(
  Percentile    = c("Q25", "Q50", "Q75", "Q90"),
  Step_Length_m = round(quantile(steps_all$sl_, c(.25, .5, .75, .9),
                                 na.rm = TRUE), 1)
)


## Step 2b: Resampling Diagnostics

Per-individual retention before and after the 15 minute resampling.
Animals with sub-15 minute burst-pair fixes (e.g. 027_Agat_13969,
039_Pete_12272) show retention around 60 to 70 percent. That is
expected deduplication of redundant secondary fixes, not biological
data loss. The methods chapter documents this.


In [ ]:
cat("== 2b. Resampling loss diagnostics ==\n")

n_raw       <- nrow(tel)
n_resampled <- nrow(trk_resampled)
n_steps     <- nrow(steps_all)

summary_tbl <- data.frame(
  Stage   = c("Raw fixes (post-QC)",
              "Fixes kept after 15 min resampling",
              "Steps formed from resampled fixes"),
  Count   = c(n_raw, n_resampled, n_steps),
  Percent = round(100 * c(n_raw, n_resampled, n_steps) / n_raw, 1)
)
print(summary_tbl, row.names = FALSE)

cat(sprintf("\n  Fixes dropped by resampling: %d (%.1f %%)\n",
            n_raw - n_resampled,
            100 * (n_raw - n_resampled) / n_raw))

per_ind <- tel %>%
  count(id, name = "n_raw") %>%
  left_join(trk_resampled %>% count(id, name = "n_resampled"), by = "id") %>%
  mutate(
    n_resampled = tidyr::replace_na(n_resampled, 0L),
    dropped     = n_raw - n_resampled,
    pct_kept    = round(100 * n_resampled / n_raw, 1)
  ) %>%
  arrange(pct_kept)

cat("\nPer-individual retention (sorted, lowest first):\n")
print(per_ind, row.names = FALSE)


## Step 3: Classify Steps as In-Patch or In-Matrix

Wild boar alternate between two movement modes. Only in-matrix steps,
i.e. directed transit between patches, are used to fit the iSSF and
generate the resistance surface.

| Mode | Biology | GPS signal |
|---|---|---|
| In-patch | Resting, wallowing, foraging within a patch | Short or zero steps |
| In-matrix | Directed transit between patches | Long steps, low turning angles |

### Classification rules

The classifier only flags in-matrix steps explicitly. Anything that
fails all of these tests is left at the in-patch default. The third
rule is a multi-step rule and reduces false positives from isolated
long steps caused by GPS noise (Benhamou 2014).

| # | Condition | Label | Rationale |
|---|---|---|---|
| 1 | `sl_ > 250 m` | in-matrix | Long-distance directed transit (Podgorski et al. 2013) |
| 2 | `sl_ > 150 m` and `abs(ta_) < pi/4` | in-matrix | Directed medium-distance movement (Clontz et al. 2021) |
| 3 | 3 or more consecutive steps in the same burst with `sl_ > 100 m` and `abs(ta_) < pi/4` | in-matrix | Sustained directional run (Benhamou 2014) |
| -- | Otherwise | in-patch | Conservative default |

Runs are evaluated per `(id, burst_)` so gaps in the GPS schedule do
not chain unrelated steps together.


In [ ]:
cat("== 3. Classifying steps ==\n")

# Rule 3 parameters for sustained directionality.
MIN_DIRECTED_SL_M <- 100   # Minimum step length per step in a directional run
MIN_DIRECTED_RUN  <- 3     # Minimum number of consecutive directional steps

# Per-step directional flag: long-enough step AND straight-enough turn.
# Evaluated independently of run length, the run check happens next.
steps_classified <- steps_all %>%
  arrange(id, burst_, t1_) %>%
  group_by(id, burst_) %>%
  mutate(
    directed = !is.na(sl_) & !is.na(ta_) &
               sl_ > MIN_DIRECTED_SL_M & abs(ta_) < (pi / 4),
    # Run-length id within the burst (changes every time `directed` flips).
    run_id = {
      r <- rle(directed)
      rep(seq_along(r$lengths), r$lengths)
    }
  ) %>%
  group_by(id, burst_, run_id) %>%
  mutate(run_len = n()) %>%
  ungroup() %>%
  mutate(
    sustained_directional = directed & run_len >= MIN_DIRECTED_RUN,
    rule_applied = case_when(
      sl_ > 250
        ~ "Rule 1: sl_ > 250 m (in-matrix)",
      sl_ > 150 & !is.na(ta_) & abs(ta_) < (pi / 4)
        ~ "Rule 2: sl_ > 150 m and abs(ta_) < pi/4 (in-matrix)",
      sustained_directional
        ~ "Rule 3: >=3 consecutive sl_ > 100 m and abs(ta_) < pi/4 (in-matrix)",
      TRUE
        ~ "Default: in-patch"
    ),
    state_label = case_when(
      sl_ > 250                                      ~ "in-matrix",
      sl_ > 150 & !is.na(ta_) & abs(ta_) < (pi / 4)  ~ "in-matrix",
      sustained_directional                          ~ "in-matrix",
      TRUE                                           ~ "in-patch"
    )
  )

in_patch  <- steps_classified %>% filter(state_label == "in-patch")
in_matrix <- steps_classified %>% filter(state_label == "in-matrix")

write.csv(in_patch,  file.path(OUT_DIR, "WildBoar_InPatch_Steps.csv"),  row.names = FALSE)
write.csv(in_matrix, file.path(OUT_DIR, "WildBoar_InMatrix_Steps.csv"), row.names = FALSE)

cat("\nOverall classification:\n")
tbl_state <- table(steps_classified$state_label)
print(data.frame(
  State   = names(tbl_state),
  Count   = as.integer(tbl_state),
  Percent = round(100 * as.integer(tbl_state) / sum(tbl_state), 1)
))

cat("\nPer-rule breakdown:\n")
tbl_rule <- table(steps_classified$rule_applied)
rule_df <- data.frame(
  Rule    = names(tbl_rule),
  Count   = as.integer(tbl_rule),
  Percent = round(100 * as.integer(tbl_rule) / sum(tbl_rule), 1)
)
rule_order <- c(
  "Rule 1: sl_ > 250 m (in-matrix)",
  "Rule 2: sl_ > 150 m and abs(ta_) < pi/4 (in-matrix)",
  "Rule 3: >=3 consecutive sl_ > 100 m and abs(ta_) < pi/4 (in-matrix)",
  "Default: in-patch"
)
rule_df <- rule_df[match(rule_order, rule_df$Rule), ]
rule_df <- rule_df[!is.na(rule_df$Rule), ]
print(rule_df, row.names = FALSE)


## Step 4: Prepare Spatial Covariates (Aargau)

Build a 25 m raster stack covering Canton Aargau plus a 2 km
processing buffer. Every layer below is computed on the same
`master_grid`, so they are pixel-aligned by construction.

| Layer | Source | Wild boar relevance |
|---|---|---|
| Forest binary | SwissTLM3D `tlm_bb_bodenbedeckung` | Resting cover, core patch |
| Forest density at 50 m | Focal mean | Stepping-stone quality |
| Distance to forest edge | `terra::distance()` | Edge-foraging behaviour |
| Slope and TRI | DHM25 | Energy cost, habitat heterogeneity |
| Distance to water | SwissTLM3D water classes | Thermoregulation, wallowing |
| Distance to infrastructure | Roads, buildings, railways | Avoidance, barrier |
| Highway binary | Autobahn, Autostrasse | Absolute barrier |
| Wildlife passages | `Wildtierpassagen.gdb` | Connectivity enhancement |
| Large lakes (>= 1 km^2) | `tlm_gewaesser_stehendes_gewaesser` | Absolute barrier |


In [ ]:
cat("== 4a. Study area boundary ==\n")

kantone <- st_read(BOUNDARIES_PATH, layer = "tlm_kantonsgebiet", quiet = TRUE) %>%
  st_transform(2056)

aargau_poly <- kantone %>% filter(name == "Aargau")
aargau_vect <- vect(aargau_poly)

# Expand the canton bbox by BUFFER_M metres in every direction. Every
# subsequent raster uses this buffered rectangle as its extent so that
# focal() and distance() near the canton boundary still see context
# from outside.
bbox <- st_bbox(aargau_poly)
bbox["xmin"] <- bbox["xmin"] - BUFFER_M
bbox["ymin"] <- bbox["ymin"] - BUFFER_M
bbox["xmax"] <- bbox["xmax"] + BUFFER_M
bbox["ymax"] <- bbox["ymax"] + BUFFER_M

aargau_bbox_rect <- st_as_sfc(bbox)

# Build the Swiss national border by unioning all cantons, then clip
# the buffered rectangle so it does not extend into Germany.
swiss_poly           <- st_union(kantone)
aargau_bbox_buffered <- st_intersection(aargau_bbox_rect, swiss_poly)

aargau_bbox_vect <- vect(aargau_bbox_buffered)
aargau_wkt       <- st_as_text(st_geometry(aargau_bbox_buffered))

# Master grid: the canonical extent for every Aargau raster in this
# pipeline.
master_grid <- rast(ext(aargau_bbox_vect),
                    resolution = TARGET_RES_M, crs = "EPSG:2056")

cat(sprintf("  Master grid: %d rows x %d cols, res = %d m\n",
            nrow(master_grid), ncol(master_grid), TARGET_RES_M))


In [ ]:
# Visualise the Aargau study area. Shows the original buffered bbox,
# the Swiss-border-clipped extent that is actually used, and the
# canton itself, on top of the Swiss country outline.

PLOT_OUT <- "../docs/presentation/images/aargau_study_area.png"

# Wrap sfc objects in sf so they can carry a fill aesthetic for the legend.
bbox_layer   <- st_as_sf(aargau_bbox_rect)
clip_layer   <- st_as_sf(aargau_bbox_buffered)
canton_layer <- aargau_poly

p_area <- ggplot() +
  geom_sf(data = swiss_poly,
          fill = "grey97", colour = "grey55", linewidth = 0.3) +
  geom_sf(data = bbox_layer,
          aes(fill = "Original bbox + 2 km buffer"),
          colour = "#D85A30", linetype = "dashed", linewidth = 0.6,
          alpha = 0) +
  geom_sf(data = clip_layer,
          aes(fill = "Clipped to Swiss border (used)"),
          colour = "#1D6FA5", linewidth = 0.5, alpha = 0.18) +
  geom_sf(data = canton_layer,
          aes(fill = "Canton Aargau"),
          colour = "#1B6E2A", linewidth = 0.7, alpha = 0.35) +
  scale_fill_manual(
    name   = NULL,
    values = c(
      "Original bbox + 2 km buffer"    = "white",
      "Clipped to Swiss border (used)" = "#1D6FA5",
      "Canton Aargau"                  = "#2A9D3F"
    ),
    guide  = guide_legend(override.aes = list(
      colour   = c("#1B6E2A", "#1D6FA5", "#D85A30"),
      linetype = c("solid",   "solid",   "dashed"),
      alpha    = c(0.35,      0.18,      0)
    ))
  ) +
  coord_sf(
    xlim   = c(st_bbox(bbox_layer)["xmin"] - 4000,
               st_bbox(bbox_layer)["xmax"] + 4000),
    ylim   = c(st_bbox(bbox_layer)["ymin"] - 4000,
               st_bbox(bbox_layer)["ymax"] + 4000),
    crs    = 2056,
    expand = FALSE
  ) +
  labs(
    title    = "Aargau study area",
    subtitle = sprintf(
      "Canton bbox + %d m buffer, clipped to the Swiss national border (excludes Germany)",
      BUFFER_M
    ),
    x = NULL, y = NULL
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title.position = "plot",
    legend.position     = "bottom",
    panel.grid          = element_blank(),
    axis.text           = element_text(size = 8)
  )

print(p_area)

ggsave(PLOT_OUT, p_area, width = 8, height = 7, dpi = 200, bg = "white")
cat(sprintf("  Saved: %s\n", PLOT_OUT))


In [ ]:
cat("== 4b. Forest cover, density and edge distance ==\n")

# Helper: rasterize a vector layer with touches = TRUE. This matters
# for narrow linear features, otherwise thin polygons miss the
# raster grid entirely.
stamp <- function(r, sf_obj, val = 1) {
  if (!is.null(sf_obj) && nrow(sf_obj) > 0)
    rasterize(vect(sf_obj), r, field = val, update = TRUE, touches = TRUE)
  else r
}

# Load land cover once and reuse for the forest mask, clipped to the
# buffered rectangle.
bodenbedeckung <- st_read(TLM_PATH, layer = "tlm_bb_bodenbedeckung",
                          wkt_filter = aargau_wkt, quiet = TRUE)

# Forest binary.
forest_sf <- bodenbedeckung %>% filter(objektart %in% FOREST_CLASSES)
r_forest  <- rasterize(vect(forest_sf), master_grid,
                       field = 1, background = 0, touches = TRUE)
r_forest  <- mask(r_forest, aargau_bbox_vect)
names(r_forest) <- "forest"

# Forest density: proportion of forest within a 50 m radius. Build a
# Boolean focal mask (1 inside the circle, NA outside) so that
# focal(mean) returns the exact proportion of forest within the disk.
cat("  Calculating 50 m forest density.\n")
fw <- focalMat(r_forest, 50, "circle")
fw[fw > 0]  <- 1
fw[fw == 0] <- NA

r_forest_dens <- focal(r_forest, w = fw, fun = "mean", na.rm = TRUE)
r_forest_dens <- mask(r_forest_dens, aargau_bbox_vect)
names(r_forest_dens) <- "forest_density"

# Unsigned distance to the forest edge. Outside forest the value is
# the distance to the nearest forest pixel; inside forest it is the
# distance to the nearest non-forest pixel (depth into the forest).
# Both are positive, giving an unsigned distance-to-edge raster.
cat("  Calculating distance to forest edge.\n")
dist_outside <- distance(ifel(r_forest == 1, 1, NA))
dist_inside  <- distance(ifel(r_forest == 0, 1, NA))
r_dist_forest <- ifel(r_forest == 1, dist_inside, dist_outside)
r_dist_forest <- mask(r_dist_forest, aargau_bbox_vect)
names(r_dist_forest) <- "dist_forest_edge"

writeRaster(r_forest,      file.path(TEMP_DIR, "forest.tif"),           overwrite = TRUE)
writeRaster(r_forest_dens, file.path(TEMP_DIR, "forest_density.tif"),   overwrite = TRUE)
writeRaster(r_dist_forest, file.path(TEMP_DIR, "dist_forest_edge.tif"), overwrite = TRUE)

cat("  Forest layers saved.\n")


In [ ]:
cat("== 4c. Terrain: slope and TRI from DHM25 ==\n")

# DHM25 has no embedded CRS, assign LV03 manually.
dem_raw      <- rast(DEM_PATH)
crs(dem_raw) <- DEM_CRS

# Transform the buffered bbox to LV03 to crop the raw data efficiently
# before computing metrics.
buffer_lv03 <- st_transform(aargau_bbox_buffered, 21781)
dem_cropped <- crop(dem_raw, ext(vect(buffer_lv03)))

cat("  Calculating slope and TRI on native resolution.\n")
slope_raw <- terrain(dem_cropped, v = "slope", unit = "degrees")
tri_raw   <- terrain(dem_cropped, v = "TRI")

# Project the computed metrics onto the LV95 master grid. Bilinear is
# the correct method here because slope and TRI are continuous.
cat("  Reprojecting to LV95 master grid.\n")
r_slope <- project(slope_raw, master_grid, method = "bilinear")
r_tri   <- project(tri_raw,   master_grid, method = "bilinear")

r_slope <- mask(r_slope, aargau_bbox_vect); names(r_slope) <- "slope"
r_tri   <- mask(r_tri,   aargau_bbox_vect); names(r_tri)   <- "tri"

writeRaster(r_slope, file.path(TEMP_DIR, "slope.tif"), overwrite = TRUE)
writeRaster(r_tri,   file.path(TEMP_DIR, "tri.tif"),   overwrite = TRUE)

cat("  Terrain layers saved.\n")


In [ ]:
cat("== 4d. Distance to water and large lake mask ==\n")

stehendes_gewaesser <- st_read(TLM_PATH,
                               layer       = "tlm_gewaesser_stehendes_gewaesser",
                               wkt_filter  = aargau_wkt, quiet = TRUE)
fliessgewaesser     <- st_read(TLM_PATH,
                               layer       = "tlm_gewaesser_fliessgewaesser",
                               wkt_filter  = aargau_wkt, quiet = TRUE)

# Binary water mask: NA outside water, 1 on every water pixel
# (lakes plus rivers). This is the layer used for the distance
# computation and for the soft-barrier burn-in.
r_water_bin <- rast(master_grid); r_water_bin[] <- NA

# Helper: convert TLM3D water-body line segments into solid polygons.
# TLM3D distributes lake shorelines as line strings grouped by gwl_nr.
# We union the segments per water body, line-merge, then cast to
# POLYGON. Casting to MULTILINESTRING first is necessary because
# st_union() may return LINESTRING for single-segment groups, which
# st_line_merge() rejects.
make_solid_polys <- function(lines_sf) {
  if (is.null(lines_sf) || nrow(lines_sf) == 0) return(NULL)
  fused <- lines_sf %>%
    group_by(gwl_nr) %>%
    summarise(geometry = st_union(geom), .groups = "drop")
  fused  <- suppressWarnings(st_cast(fused, "MULTILINESTRING"))
  merged <- st_line_merge(fused)
  polys       <- suppressWarnings(st_cast(merged, "POLYGON"))
  polys_valid <- st_make_valid(polys)
  st_collection_extract(polys_valid, "POLYGON")
}

# Standing water (lakes and islands).
lakes_polys_all <- NULL
if (nrow(stehendes_gewaesser) > 0) {
  stehend_2d    <- st_zm(stehendes_gewaesser, drop = TRUE, what = "ZM")
  lakes_lines   <- stehend_2d %>% filter(objektart == "See")
  islands_lines <- stehend_2d %>% filter(objektart == "Seeinsel")

  lakes_polys   <- make_solid_polys(lakes_lines)
  islands_polys <- make_solid_polys(islands_lines)

  if (!is.null(lakes_polys) && nrow(lakes_polys) > 0) {
    lakes_polys_all <- lakes_polys
    if (!is.null(islands_polys) && nrow(islands_polys) > 0) {
      lakes_polys_all <- st_difference(lakes_polys, st_union(islands_polys))
    }
    r_water_bin <- rasterize(vect(lakes_polys_all), r_water_bin,
                             field = 1, update = TRUE)
  }
}

# Flowing water with the ecological filter applied.
if (nrow(fliessgewaesser) > 0) {
  fliess_2d       <- st_zm(fliessgewaesser, drop = TRUE, what = "ZM")
  fliess_filtered <- fliess_2d %>%
    filter(
      objektart == "Fliessgewaesser",
      verlauf   %in% c("Oberirdisch", "Bruecke", "Wasserfall",
                       "Brücke")
    )
  if (nrow(fliess_filtered) > 0) {
    r_water_bin <- rasterize(vect(st_geometry(fliess_filtered)), r_water_bin,
                             field = 1, update = TRUE, touches = TRUE)
  }
}

# Distance to nearest water pixel.
cat("  Calculating distance to water.\n")
r_dist_water <- distance(r_water_bin)
r_dist_water <- mask(r_dist_water, aargau_bbox_vect)
names(r_dist_water) <- "dist_water"
writeRaster(r_dist_water, file.path(TEMP_DIR, "dist_water.tif"), overwrite = TRUE)

# Large lakes only. Lakes whose area exceeds LAKE_MIN_AREA_M2 become
# absolute barriers in Steps 6 and 7c. Smaller water bodies stay in
# r_water_bin only and act as soft barriers there.
r_large_lakes <- rast(master_grid); r_large_lakes[] <- NA
if (!is.null(lakes_polys_all) && nrow(lakes_polys_all) > 0) {
  areas <- as.numeric(st_area(lakes_polys_all))
  large_lakes_polys <- lakes_polys_all[areas > LAKE_MIN_AREA_M2, ]
  if (nrow(large_lakes_polys) > 0) {
    r_large_lakes <- rasterize(vect(large_lakes_polys), r_large_lakes,
                               field = 1, update = TRUE)
    cat(sprintf("  Large lakes (area > %.1f km^2): %d\n",
                LAKE_MIN_AREA_M2 / 1e6, nrow(large_lakes_polys)))
  }
}
names(r_large_lakes) <- "large_lakes"
writeRaster(r_large_lakes, file.path(TEMP_DIR, "large_lakes.tif"),
            overwrite = TRUE)

cat("  Water distance and large lake layers saved.\n")


In [ ]:
cat("== 4e. Infrastructure, roads, settlements and wildlife passages ==\n")

# Drop 3D vertices and filter underground features. Primary check is
# STUFE < 0 (numeric coded value from the TLM3D spec). Secondary
# check is KUNSTBAUTE in the configured underground list.
prep_above_ground <- function(sf_obj) {
  if (is.null(sf_obj) || nrow(sf_obj) == 0) return(sf_obj)
  x <- sf::st_zm(sf_obj, drop = TRUE, what = "ZM")
  if ("stufe" %in% names(x)) {
    s <- suppressWarnings(as.numeric(as.character(x$stufe)))
    x <- x[is.na(s) | s >= 0, , drop = FALSE]
  }
  if ("kunstbaute" %in% names(x)) {
    x <- x[is.na(x$kunstbaute) | !(x$kunstbaute %in% KUNSTBAUTE_UNDERGROUND), ,
           drop = FALSE]
  }
  x
}

# Drop disused and specialty railways. These are not real boar
# barriers.
clean_railways <- function(sf_obj) {
  if (is.null(sf_obj) || nrow(sf_obj) == 0) return(sf_obj)
  if ("ausser_betrieb" %in% names(sf_obj)) {
    sf_obj <- sf_obj[is.na(sf_obj$ausser_betrieb) |
                     sf_obj$ausser_betrieb != "Wahr", , drop = FALSE]
  }
  for (flag in c("standseilbahn", "zahnradbahn", "museumsbahn")) {
    if (flag %in% names(sf_obj)) {
      sf_obj <- sf_obj[is.na(sf_obj[[flag]]) |
                       sf_obj[[flag]] != "Wahr", , drop = FALSE]
    }
  }
  sf_obj
}

# Load required TLM layers (3D dropped, underground filtered). These
# always exist in the standard TLM file, so no try/catch defence is
# warranted.
strassen <- prep_above_ground(
  st_read(TLM_PATH, layer = "tlm_strassen_strasse",
          wkt_filter = aargau_wkt, quiet = TRUE)
)
eisenbahn <- clean_railways(prep_above_ground(
  st_read(TLM_PATH, layer = "tlm_oev_eisenbahn",
          wkt_filter = aargau_wkt, quiet = TRUE)
))
siedlung_sf <- st_read(TLM_PATH, layer = SETTLEMENT_LAYER,
                       wkt_filter = aargau_wkt, quiet = TRUE) %>%
  st_zm(drop = TRUE, what = "ZM") %>%
  filter(objektart %in% SETTLEMENT_CLASSES)
cat(sprintf("  Settlement polygons: %d\n", nrow(siedlung_sf)))

# Split roads by class.
fenced_roads <- strassen %>% filter(objektart %in% FENCED_ROAD_CLASSES)
main_roads   <- strassen %>% filter(objektart %in% MAINROAD_CLASSES)
sec_roads    <- strassen %>% filter(objektart %in% SECONDARY_ROAD_CLASSES)
cat(sprintf("  After filtering: fenced=%d, main=%d, secondary=%d, rail=%d\n",
            nrow(fenced_roads), nrow(main_roads), nrow(sec_roads),
            nrow(eisenbahn)))

# Per-class distance rasters used as iSSF covariates.
r_fenced_bin  <- rast(master_grid); r_fenced_bin[]  <- NA
r_fenced_bin  <- stamp(r_fenced_bin, fenced_roads)
r_dist_fenced <- mask(distance(r_fenced_bin), aargau_bbox_vect)
names(r_dist_fenced) <- "dist_fenced_road"

r_main_bin  <- rast(master_grid); r_main_bin[]  <- NA
r_main_bin  <- stamp(r_main_bin, main_roads)
r_dist_main <- mask(distance(r_main_bin), aargau_bbox_vect)
names(r_dist_main) <- "dist_main_road"

# Secondary roads are used as a graded iSSF covariate only. They are
# NOT burned in as an absolute barrier because boar routinely cross
# 2 to 4 m forest tracks.
r_sec_bin  <- rast(master_grid); r_sec_bin[]  <- NA
r_sec_bin  <- stamp(r_sec_bin, sec_roads)
r_dist_sec <- mask(distance(r_sec_bin), aargau_bbox_vect)
names(r_dist_sec) <- "dist_secondary_road"

# Settlements as built-up polygons, not individual buildings.
r_settle_bin <- rast(master_grid); r_settle_bin[] <- NA
r_settle_bin <- rasterize(vect(st_make_valid(siedlung_sf)), r_settle_bin,
                          field = 1, update = TRUE, touches = TRUE)
r_dist_settle <- mask(distance(r_settle_bin), aargau_bbox_vect)
names(r_dist_settle) <- "dist_settlement"

# Railways.
r_rail_bin  <- rast(master_grid); r_rail_bin[]  <- NA
r_rail_bin  <- stamp(r_rail_bin, eisenbahn)
r_dist_rail <- mask(distance(r_rail_bin), aargau_bbox_vect)
names(r_dist_rail) <- "dist_railway"

# Combined infrastructure mask used by Step 6 as a soft-barrier
# burn-in. Secondary roads are intentionally excluded.
r_infra <- rast(master_grid); r_infra[] <- NA
r_infra <- stamp(r_infra, fenced_roads)
r_infra <- stamp(r_infra, main_roads)
r_infra <- stamp(r_infra, eisenbahn)
r_infra <- rasterize(vect(st_make_valid(siedlung_sf)), r_infra,
                     field = 1, update = TRUE, touches = TRUE)

# Highway binary used by Step 6 as an absolute barrier (NA).
r_highway <- rast(master_grid); r_highway[] <- 0
r_highway <- stamp(r_highway, fenced_roads)
r_highway <- mask(r_highway, aargau_bbox_vect)
names(r_highway) <- "highway"

# Wildlife passages. Required input: the pipeline cannot honour the
# Step 6 'passages override absolute barriers' rule without it.
passages_sf <- st_read(PASSAGES_GDB, layer = PASSAGES_LAYER, quiet = TRUE) %>%
  st_zm(drop = TRUE, what = "ZM") %>%
  st_transform(2056) %>%
  st_filter(aargau_bbox_buffered)
r_passages <- rast(master_grid); r_passages[] <- 0
r_passages <- stamp(r_passages, st_buffer(passages_sf, dist = 30))
names(r_passages) <- "passages"
cat(sprintf("  Wildlife passages: %d (buffered to 30 m)\n",
            nrow(passages_sf)))

writeRaster(r_dist_fenced, file.path(TEMP_DIR, "dist_fenced_road.tif"),    overwrite = TRUE)
writeRaster(r_dist_main,   file.path(TEMP_DIR, "dist_main_road.tif"),      overwrite = TRUE)
writeRaster(r_dist_sec,    file.path(TEMP_DIR, "dist_secondary_road.tif"), overwrite = TRUE)
writeRaster(r_dist_settle, file.path(TEMP_DIR, "dist_settlement.tif"),     overwrite = TRUE)
writeRaster(r_dist_rail,   file.path(TEMP_DIR, "dist_railway.tif"),        overwrite = TRUE)
writeRaster(r_highway,     file.path(TEMP_DIR, "highway.tif"),             overwrite = TRUE)
writeRaster(r_passages,    file.path(TEMP_DIR, "passages.tif"),            overwrite = TRUE)

cat("  Infrastructure and passage layers saved.\n")


In [ ]:
cat("== 4f. Assemble covariate stack ==\n")

cov_stack <- c(r_forest, r_forest_dens, r_dist_forest,
               r_slope, r_tri, r_dist_water,
               r_dist_fenced, r_dist_main, r_dist_sec,
               r_dist_settle, r_dist_rail)
names(cov_stack) <- c("forest", "forest_density", "dist_forest_edge",
                      "slope", "tri", "dist_water",
                      "dist_fenced_road", "dist_main_road", "dist_secondary_road",
                      "dist_settlement", "dist_railway")

writeRaster(cov_stack, file.path(TEMP_DIR, "covariate_stack.tif"),
            overwrite = TRUE)

cat(sprintf("  Stack: %d layers at %d m resolution\n",
            nlyr(cov_stack), TARGET_RES_M))
print(names(cov_stack))


## Step 5: Integrated Step-Selection Function (iSSF)

Fit a conditional logistic regression on in-matrix steps only,
comparing each observed step to `N_RANDOM_STEPS` random alternatives
drawn from the empirical step-length and turning-angle distributions.

Covariates are z-standardised so the beta coefficients are directly
comparable.

- Positive beta means the habitat is preferred during transit, which
  translates into lower resistance.
- Negative beta means the habitat is avoided during transit, which
  translates into higher resistance.

The scaling parameters (mean and standard deviation per covariate)
are saved. They are reused in Steps 6 and 7 so the Zurich rasters end
up scaled identically to the Aargau training data.


In [ ]:
cat("== 5. Fitting iSSF on in-matrix steps ==\n")

# Prepare clean in-matrix steps.
in_matrix_clean <- in_matrix %>%
  filter(!is.na(sl_), !is.na(ta_), sl_ > 0)
class(in_matrix_clean) <- c("steps_xyt", "steps_xy", "data.frame")
cat(sprintf("  In-matrix steps: %d\n", nrow(in_matrix_clean)))

# Generate random control steps.
issf_data <- in_matrix_clean %>%
  random_steps(n_control = N_RANDOM_STEPS) %>%
  mutate(
    log_sl = log(sl_ + 1),   # Movement kernel term
    cos_ta = cos(ta_)        # Directional persistence term
  )

# Extract covariate values at step endpoints. The telemetry CSV may
# carry either LV03 or LV95 coordinates, so detect the native CRS
# from the magnitude of x2_ and then project to the raster CRS for
# the extract.
cat("  Extracting covariates at step endpoints.\n")
is_lv03    <- mean(issf_data$x2_, na.rm = TRUE) < 1000000
native_crs <- ifelse(is_lv03, "EPSG:21781", "EPSG:2056")
cat(sprintf("  Detected native point CRS: %s\n", native_crs))

pts           <- terra::vect(issf_data, geom = c("x2_", "y2_"),
                             crs = native_crs)
pts_projected <- terra::project(pts, terra::crs(cov_stack))
ext_val       <- terra::extract(cov_stack, pts_projected)

issf_data <- cbind(issf_data, ext_val[, -1])
issf_data <- issf_data %>%
  filter(complete.cases(across(all_of(names(cov_stack)))))
cat(sprintf("  Valid steps after extraction: %d\n", nrow(issf_data)))
if (nrow(issf_data) == 0) stop("No valid steps. The extents do not overlap.")

# Identify and z-standardise the continuous covariates. Binary forest
# is handled separately as a factor.
continuous_covs <- c("forest_density", "dist_forest_edge",
                     "slope", "tri", "dist_water",
                     "dist_fenced_road", "dist_main_road", "dist_secondary_road",
                     "dist_settlement", "dist_railway")
available_covs  <- intersect(continuous_covs, names(issf_data))

# Drop covariates with zero variance (constant raster or no data).
valid_covs <- Filter(function(cov) {
  s <- sd(issf_data[[cov]], na.rm = TRUE)
  if (is.na(s) || s == 0) {
    cat(sprintf("  Note: '%s' dropped (zero variance)\n", cov))
    FALSE
  } else TRUE
}, available_covs)

# Save scaling parameters. These must be reused for the prediction
# rasters in Steps 6 and 7 so Zurich is scaled with Aargau statistics.
scale_params <- issf_data %>%
  summarise(across(all_of(valid_covs),
                   list(mean = ~ mean(.x, na.rm = TRUE),
                        sd   = ~ sd(.x,   na.rm = TRUE))))

issf_data <- issf_data %>%
  mutate(across(all_of(valid_covs),
                ~ (.x - mean(.x, na.rm = TRUE)) / sd(.x, na.rm = TRUE),
                .names = "{.col}_sc"))

issf_data$forest_fct <- as.factor(issf_data$forest)

# Build the model formula dynamically.
scaled_covs <- paste0(valid_covs, "_sc")
frm <- as.formula(paste(
  "case_ ~",
  paste(c("forest_fct", "log_sl", "cos_ta", scaled_covs, "strata(step_id_)"),
        collapse = " + ")
))
cat("  Formula:\n"); print(frm)

# Final NA drop.
issf_model_data <- issf_data %>%
  drop_na(all_of(c("case_", "forest_fct", "log_sl", "cos_ta", scaled_covs)))
cat(sprintf("  Rows used in model: %d\n", nrow(issf_model_data)))
if (nrow(issf_model_data) == 0) stop("No rows remaining after NA drop.")

# Fit conditional logistic regression.
cat("  Fitting clogit.\n")
m_issf <- survival::clogit(frm, data = issf_model_data, method = "efron")
cat("  Model fitted.\n")
print(summary(m_issf))

# Save coefficients and a forest plot of the selection coefficients.
coef_df <- broom::tidy(m_issf, conf.int = TRUE) %>%
  filter(!grepl("log_sl|cos_ta|step_id", term)) %>%
  arrange(desc(abs(estimate)))

write.csv(coef_df, file.path(OUT_DIR, "issf_coefficients.csv"),
          row.names = FALSE)

p_coef <- ggplot(coef_df,
                 aes(x = estimate, y = reorder(term, estimate),
                     xmin = conf.low, xmax = conf.high)) +
  geom_vline(xintercept = 0, linetype = "dashed", colour = "grey60") +
  geom_pointrange(colour = "#1D6FA5", size = 0.7, linewidth = 0.9) +
  labs(x = "Selection coefficient beta (positive = preferred during transit)",
       y = NULL,
       title = "iSSF: Wild Boar In-Matrix Habitat Selection",
       subtitle = "Canton Aargau, in-matrix steps only") +
  theme_minimal(base_size = 11)

ggsave(file.path(OUT_DIR, "issf_coefficients.png"), p_coef,
       width = 8, height = 5, dpi = 200, bg = "white")
cat("  Saved: issf_coefficients.csv and issf_coefficients.png\n")


## Step 6: Resistance Surface for Aargau (Keeley et al. 2016)

Three-stage conversion from iSSF coefficients to a resistance raster:

1. Linear predictor `eta = sum(beta_k * x_k_scaled)`.
2. Habitat suitability `S = (eta - eta_min) / (eta_max - eta_min)` in
   [0, 1].
3. Resistance `R = exp(KEELEY_C * (1 - S))`. With `C = 4` this gives
   `R` in [1, 54.6].

The final resistance raster then receives three classes of burn-in:

- Soft barriers (main roads, railways, settlements, small water
  bodies) get high but finite resistance values. Boar can still
  cross them at high cost.
- Absolute barriers (highways, large lakes, and, in Zurich,
  permanent ASP fences) are written as `NA`, which the downstream
  QGIS plugin treats as impassable.
- Wildlife passages are written last and override every other
  burn-in, including absolute barriers, so legal crossings are
  always permeable.

The output is finally masked to the exact canton polygon. The 2 km
buffer ring is dropped at this stage because it carries residual edge
effects and is not part of the deliverable.


In [ ]:
cat("== 6. Building Aargau resistance raster ==\n")

beta  <- coef(m_issf)
R_max <- exp(KEELEY_C)   # ~ 54.6
R_min <- 1

# Helpers.
get_beta <- function(name) {
  b <- beta[name]
  if (is.null(b) || is.na(b)) 0 else as.numeric(b)
}
scale_raster <- function(r, varname) {
  mn <- as.numeric(scale_params[[paste0(varname, "_mean")]])
  sd <- as.numeric(scale_params[[paste0(varname, "_sd")]])
  (r - mn) / sd
}

# Scale rasters using Aargau training parameters.
r_forest_dens_sc      <- scale_raster(r_forest_dens, "forest_density")
r_dist_forest_edge_sc <- scale_raster(r_dist_forest, "dist_forest_edge")
r_dist_water_sc       <- scale_raster(r_dist_water,  "dist_water")
r_dist_fenced_sc      <- scale_raster(r_dist_fenced, "dist_fenced_road")
r_dist_main_sc        <- scale_raster(r_dist_main,   "dist_main_road")
r_dist_sec_sc         <- scale_raster(r_dist_sec,    "dist_secondary_road")
r_dist_settle_sc      <- scale_raster(r_dist_settle, "dist_settlement")
r_dist_rail_sc        <- scale_raster(r_dist_rail,   "dist_railway")

# Linear predictor eta.
eta <- (r_forest_dens_sc      * get_beta("forest_density_sc"))      +
       (r_dist_forest_edge_sc * get_beta("dist_forest_edge_sc"))    +
       (r_dist_water_sc       * get_beta("dist_water_sc"))          +
       (r_dist_fenced_sc      * get_beta("dist_fenced_road_sc"))    +
       (r_dist_main_sc        * get_beta("dist_main_road_sc"))      +
       (r_dist_sec_sc         * get_beta("dist_secondary_road_sc")) +
       (r_dist_settle_sc      * get_beta("dist_settlement_sc"))     +
       (r_dist_rail_sc        * get_beta("dist_railway_sc"))

# Store the eta range; Step 7 reuses it for the Zurich transfer.
eta_min <- global(eta, "min", na.rm = TRUE)[[1]]
eta_max <- global(eta, "max", na.rm = TRUE)[[1]]

# Habitat suitability S in [0, 1].
S <- (eta - eta_min) / (eta_max - eta_min)

# Keeley exponential transformation.
R <- exp(KEELEY_C * (1 - S))

# Burn in barriers. Order matters because each line overwrites
# previous assignments where masks overlap.
#
#   1. Soft barriers first: small water bodies, then r_infra which
#      includes highways, main roads, railways, and settlements.
#   2. Absolute barriers next: highways and large lakes become NA.
#      These overwrite the R_max assignments from r_infra.
#   3. Passages last: even cells marked NA above can be crossed at
#      wildlife passages, so passages override everything.
R[!is.na(r_water_bin) & r_water_bin == 1 & is.na(r_large_lakes)] <- R_max * 0.9
R[r_infra == 1]                                                  <- R_max
R[r_highway == 1]                                                <- NA
R[!is.na(r_large_lakes) & r_large_lakes == 1]                    <- NA
R[r_passages == 1]                                               <- R_min

# Mask to the exact canton polygon (not the buffered rectangle). The
# 2 km buffer was needed for edge-effect-free covariate computation,
# but the deliverable should not contain that ring.
S <- mask(S, aargau_vect); names(S) <- "habitat_suitability"
R <- mask(R, aargau_vect); names(R) <- "resistance"

writeRaster(S, file.path(OUT_DIR, "Habitat_Suitability_Wildboar_Aargau.tif"),
            overwrite = TRUE, NAflag = -9999)
writeRaster(R, file.path(OUT_DIR, "Resistance_Keeley_Wildboar_Aargau_25m.tif"),
            overwrite = TRUE, NAflag = -9999)

cat(sprintf("  Resistance range (finite cells): [%.2f, %.2f]\n",
            global(R, "min", na.rm = TRUE)[[1]],
            global(R, "max", na.rm = TRUE)[[1]]))
cat(sprintf("  Impassable cells (NA): %d\n",
            global(is.na(R), "sum", na.rm = TRUE)[[1]]))
cat("  Aargau resistance and suitability rasters saved.\n")


## Step 7: Transfer Resistance Surface to Canton Zurich

Rebuild identical covariate layers for Zurich and then apply the
Aargau model.

Golden rule. Zurich rasters are scaled with Aargau means and standard
deviations from `scale_params`, and normalised with Aargau's eta
range (`eta_min`, `eta_max`). This makes the resistance values
behaviourally calibrated and ensures Aargau and Zurich form a
seamless surface.

Zurich-specific addition. Permanent ASP fences from `Dauerzaeune.shp`
(VetAmt Zurich) are burned in as absolute barriers (`NA`). Wild boar
cannot pass through woven-wire ASP fencing.


In [ ]:
cat("== 7a. Zurich boundary and master grid ==\n")

zurich_poly <- kantone %>% filter(name == "Zürich")
zurich_vect <- vect(zurich_poly)

# Buffered rectangle around Canton Zurich (same procedure as Aargau).
bbox_zh <- st_bbox(zurich_poly)
bbox_zh["xmin"] <- bbox_zh["xmin"] - BUFFER_M
bbox_zh["ymin"] <- bbox_zh["ymin"] - BUFFER_M
bbox_zh["xmax"] <- bbox_zh["xmax"] + BUFFER_M
bbox_zh["ymax"] <- bbox_zh["ymax"] + BUFFER_M

zurich_bbox_rect     <- st_as_sfc(bbox_zh)
zurich_bbox_buffered <- st_intersection(zurich_bbox_rect, swiss_poly)

zurich_bbox_vect <- vect(zurich_bbox_buffered)
zurich_wkt       <- st_as_text(st_geometry(zurich_bbox_buffered))

master_grid_zh <- rast(ext(zurich_bbox_vect),
                       resolution = TARGET_RES_M, crs = "EPSG:2056")

cat(sprintf("  Zurich master grid: %d rows x %d cols\n",
            nrow(master_grid_zh), ncol(master_grid_zh)))


In [ ]:
cat("== 7b. Zurich covariate layers ==\n")

# Forest cover, density, and edge distance.
bodenbedeckung_zh <- st_read(TLM_PATH, layer = "tlm_bb_bodenbedeckung",
                             wkt_filter = zurich_wkt, quiet = TRUE)

forest_sf_zh <- bodenbedeckung_zh %>% filter(objektart %in% FOREST_CLASSES)
r_forest_zh  <- rasterize(vect(forest_sf_zh), master_grid_zh,
                          field = 1, background = 0, touches = TRUE)
r_forest_zh  <- mask(r_forest_zh, zurich_bbox_vect)
names(r_forest_zh) <- "forest"

cat("  Calculating 50 m forest density (ZH).\n")
fw_zh <- focalMat(r_forest_zh, 50, "circle")
fw_zh[fw_zh > 0]  <- 1
fw_zh[fw_zh == 0] <- NA

r_forest_dens_zh <- focal(r_forest_zh, w = fw_zh, fun = "mean", na.rm = TRUE)
r_forest_dens_zh <- mask(r_forest_dens_zh, zurich_bbox_vect)
names(r_forest_dens_zh) <- "forest_density"

cat("  Calculating distance to forest edge (ZH).\n")
dist_outside_zh  <- distance(ifel(r_forest_zh == 1, 1, NA))
dist_inside_zh   <- distance(ifel(r_forest_zh == 0, 1, NA))
r_dist_forest_zh <- ifel(r_forest_zh == 1, dist_inside_zh, dist_outside_zh)
r_dist_forest_zh <- mask(r_dist_forest_zh, zurich_bbox_vect)
names(r_dist_forest_zh) <- "dist_forest_edge"

# Water (standing plus flowing) and the large lake mask.
stehendes_gewaesser_zh <- st_read(TLM_PATH,
                                  layer       = "tlm_gewaesser_stehendes_gewaesser",
                                  wkt_filter  = zurich_wkt, quiet = TRUE)
fliessgewaesser_zh     <- st_read(TLM_PATH,
                                  layer       = "tlm_gewaesser_fliessgewaesser",
                                  wkt_filter  = zurich_wkt, quiet = TRUE)

r_water_bin_zh <- rast(master_grid_zh); r_water_bin_zh[] <- NA

lakes_polys_all_zh <- NULL
if (nrow(stehendes_gewaesser_zh) > 0) {
  stehend_2d_zh    <- st_zm(stehendes_gewaesser_zh, drop = TRUE, what = "ZM")
  lakes_lines_zh   <- stehend_2d_zh %>% filter(objektart == "See")
  islands_lines_zh <- stehend_2d_zh %>% filter(objektart == "Seeinsel")

  lakes_polys_zh   <- make_solid_polys(lakes_lines_zh)
  islands_polys_zh <- make_solid_polys(islands_lines_zh)

  if (!is.null(lakes_polys_zh) && nrow(lakes_polys_zh) > 0) {
    lakes_polys_all_zh <- lakes_polys_zh
    if (!is.null(islands_polys_zh) && nrow(islands_polys_zh) > 0) {
      lakes_polys_all_zh <- st_difference(lakes_polys_zh,
                                          st_union(islands_polys_zh))
    }
    r_water_bin_zh <- rasterize(vect(lakes_polys_all_zh), r_water_bin_zh,
                                field = 1, update = TRUE)
  }
}

if (nrow(fliessgewaesser_zh) > 0) {
  fliess_2d_zh       <- st_zm(fliessgewaesser_zh, drop = TRUE, what = "ZM")
  fliess_filtered_zh <- fliess_2d_zh %>%
    filter(
      objektart == "Fliessgewaesser",
      verlauf   %in% c("Oberirdisch", "Bruecke", "Wasserfall",
                       "Brücke")
    )
  if (nrow(fliess_filtered_zh) > 0) {
    r_water_bin_zh <- rasterize(vect(st_geometry(fliess_filtered_zh)),
                                r_water_bin_zh,
                                field = 1, update = TRUE, touches = TRUE)
  }
}

cat("  Calculating distance to water (ZH).\n")
r_dist_water_zh <- distance(r_water_bin_zh)
r_dist_water_zh <- mask(r_dist_water_zh, zurich_bbox_vect)
names(r_dist_water_zh) <- "dist_water"

# Large lakes for Zurich (Zurichsee, Greifensee, Pfaeffikersee).
r_large_lakes_zh <- rast(master_grid_zh); r_large_lakes_zh[] <- NA
if (!is.null(lakes_polys_all_zh) && nrow(lakes_polys_all_zh) > 0) {
  areas_zh <- as.numeric(st_area(lakes_polys_all_zh))
  large_lakes_polys_zh <- lakes_polys_all_zh[areas_zh > LAKE_MIN_AREA_M2, ]
  if (nrow(large_lakes_polys_zh) > 0) {
    r_large_lakes_zh <- rasterize(vect(large_lakes_polys_zh),
                                  r_large_lakes_zh,
                                  field = 1, update = TRUE)
    cat(sprintf("  Large lakes (ZH, area > %.1f km^2): %d\n",
                LAKE_MIN_AREA_M2 / 1e6, nrow(large_lakes_polys_zh)))
  }
}
names(r_large_lakes_zh) <- "large_lakes"

# Infrastructure.
strassen_zh <- prep_above_ground(
  st_read(TLM_PATH, layer = "tlm_strassen_strasse",
          wkt_filter = zurich_wkt, quiet = TRUE)
)
eisenbahn_zh <- clean_railways(prep_above_ground(
  st_read(TLM_PATH, layer = "tlm_oev_eisenbahn",
          wkt_filter = zurich_wkt, quiet = TRUE)
))
siedlung_sf_zh <- st_read(TLM_PATH, layer = SETTLEMENT_LAYER,
                          wkt_filter = zurich_wkt, quiet = TRUE) %>%
  st_zm(drop = TRUE, what = "ZM") %>%
  filter(objektart %in% SETTLEMENT_CLASSES)

fenced_roads_zh <- strassen_zh %>% filter(objektart %in% FENCED_ROAD_CLASSES)
main_roads_zh   <- strassen_zh %>% filter(objektart %in% MAINROAD_CLASSES)
sec_roads_zh    <- strassen_zh %>% filter(objektart %in% SECONDARY_ROAD_CLASSES)

cat(sprintf("  Zurich after filtering: fenced=%d, main=%d, secondary=%d, rail=%d\n",
            nrow(fenced_roads_zh), nrow(main_roads_zh), nrow(sec_roads_zh),
            nrow(eisenbahn_zh)))

# Per-class distance rasters.
r_fenced_bin_zh  <- rast(master_grid_zh); r_fenced_bin_zh[]  <- NA
r_fenced_bin_zh  <- stamp(r_fenced_bin_zh, fenced_roads_zh)
r_dist_fenced_zh <- mask(distance(r_fenced_bin_zh), zurich_bbox_vect)
names(r_dist_fenced_zh) <- "dist_fenced_road"

r_main_bin_zh  <- rast(master_grid_zh); r_main_bin_zh[]  <- NA
r_main_bin_zh  <- stamp(r_main_bin_zh, main_roads_zh)
r_dist_main_zh <- mask(distance(r_main_bin_zh), zurich_bbox_vect)
names(r_dist_main_zh) <- "dist_main_road"

r_sec_bin_zh  <- rast(master_grid_zh); r_sec_bin_zh[]  <- NA
r_sec_bin_zh  <- stamp(r_sec_bin_zh, sec_roads_zh)
r_dist_sec_zh <- mask(distance(r_sec_bin_zh), zurich_bbox_vect)
names(r_dist_sec_zh) <- "dist_secondary_road"

r_settle_bin_zh <- rast(master_grid_zh); r_settle_bin_zh[] <- NA
r_settle_bin_zh <- rasterize(vect(st_make_valid(siedlung_sf_zh)),
                             r_settle_bin_zh,
                             field = 1, update = TRUE, touches = TRUE)
r_dist_settle_zh <- mask(distance(r_settle_bin_zh), zurich_bbox_vect)
names(r_dist_settle_zh) <- "dist_settlement"

r_rail_bin_zh  <- rast(master_grid_zh); r_rail_bin_zh[]  <- NA
r_rail_bin_zh  <- stamp(r_rail_bin_zh, eisenbahn_zh)
r_dist_rail_zh <- mask(distance(r_rail_bin_zh), zurich_bbox_vect)
names(r_dist_rail_zh) <- "dist_railway"

r_infra_zh <- rast(master_grid_zh); r_infra_zh[] <- NA
r_infra_zh <- stamp(r_infra_zh, fenced_roads_zh)
r_infra_zh <- stamp(r_infra_zh, main_roads_zh)
r_infra_zh <- stamp(r_infra_zh, eisenbahn_zh)
r_infra_zh <- rasterize(vect(st_make_valid(siedlung_sf_zh)), r_infra_zh,
                        field = 1, update = TRUE, touches = TRUE)

r_highway_zh <- rast(master_grid_zh); r_highway_zh[] <- 0
r_highway_zh <- stamp(r_highway_zh, fenced_roads_zh)
r_highway_zh <- mask(r_highway_zh, zurich_bbox_vect)
names(r_highway_zh) <- "highway"

# Wildlife passages. Required input (same reason as in Step 4e).
passages_sf_zh <- st_read(PASSAGES_GDB, layer = PASSAGES_LAYER, quiet = TRUE) %>%
  st_zm(drop = TRUE, what = "ZM") %>%
  st_transform(2056) %>%
  st_filter(zurich_bbox_buffered)
r_passages_zh <- rast(master_grid_zh); r_passages_zh[] <- 0
r_passages_zh <- stamp(r_passages_zh, st_buffer(passages_sf_zh, dist = 30))
names(r_passages_zh) <- "passages"
cat(sprintf("  Wildlife passages (ZH): %d (buffered to 30 m)\n",
            nrow(passages_sf_zh)))

# Permanent ASP fences (optional shapefile).
fences_sf_zh <- tryCatch(
  st_read(FENCE_PATH, quiet = TRUE) %>%
    st_zm(drop = TRUE, what = "ZM") %>%
    st_transform(2056) %>%
    st_filter(zurich_bbox_buffered),
  error = function(e) { message("  Note: fence layer not loaded."); NULL }
)
r_fences_zh <- rast(master_grid_zh); r_fences_zh[] <- NA
if (!is.null(fences_sf_zh) && nrow(fences_sf_zh) > 0) {
  r_fences_zh <- rasterize(vect(fences_sf_zh), r_fences_zh,
                           field = 1, update = TRUE, touches = TRUE)
}
r_fences_zh <- mask(r_fences_zh, zurich_bbox_vect)
names(r_fences_zh) <- "fences"

writeRaster(r_forest_zh,       file.path(TEMP_DIR, "forest_zurich.tif"),               overwrite = TRUE)
writeRaster(r_forest_dens_zh,  file.path(TEMP_DIR, "forest_density_zurich.tif"),       overwrite = TRUE)
writeRaster(r_dist_forest_zh,  file.path(TEMP_DIR, "dist_forest_edge_zurich.tif"),     overwrite = TRUE)
writeRaster(r_dist_water_zh,   file.path(TEMP_DIR, "dist_water_zurich.tif"),           overwrite = TRUE)
writeRaster(r_dist_fenced_zh,  file.path(TEMP_DIR, "dist_fenced_road_zurich.tif"),     overwrite = TRUE)
writeRaster(r_dist_main_zh,    file.path(TEMP_DIR, "dist_main_road_zurich.tif"),       overwrite = TRUE)
writeRaster(r_dist_sec_zh,     file.path(TEMP_DIR, "dist_secondary_road_zurich.tif"),  overwrite = TRUE)
writeRaster(r_dist_settle_zh,  file.path(TEMP_DIR, "dist_settlement_zurich.tif"),      overwrite = TRUE)
writeRaster(r_dist_rail_zh,    file.path(TEMP_DIR, "dist_railway_zurich.tif"),         overwrite = TRUE)
writeRaster(r_highway_zh,      file.path(TEMP_DIR, "highway_zurich.tif"),              overwrite = TRUE)
writeRaster(r_large_lakes_zh,  file.path(TEMP_DIR, "large_lakes_zurich.tif"),          overwrite = TRUE)
writeRaster(r_passages_zh,     file.path(TEMP_DIR, "passages_zurich.tif"),             overwrite = TRUE)
writeRaster(r_fences_zh,       file.path(TEMP_DIR, "fences_zurich.tif"),               overwrite = TRUE)

cat("  Zurich covariate layers ready and saved to data/temp.\n")


In [ ]:
cat("== 7c. Apply the Aargau model to Zurich ==\n")

# Scale Zurich rasters with Aargau training mean and sd, not with
# Zurich's own statistics.
scale_raster_zh <- function(r_zh, varname) {
  mn <- as.numeric(scale_params[[paste0(varname, "_mean")]])
  sd <- as.numeric(scale_params[[paste0(varname, "_sd")]])
  (r_zh - mn) / sd
}

r_forest_dens_sc_zh      <- scale_raster_zh(r_forest_dens_zh, "forest_density")
r_dist_forest_edge_sc_zh <- scale_raster_zh(r_dist_forest_zh, "dist_forest_edge")
r_dist_water_sc_zh       <- scale_raster_zh(r_dist_water_zh,  "dist_water")
r_dist_fenced_sc_zh      <- scale_raster_zh(r_dist_fenced_zh, "dist_fenced_road")
r_dist_main_sc_zh        <- scale_raster_zh(r_dist_main_zh,   "dist_main_road")
r_dist_sec_sc_zh         <- scale_raster_zh(r_dist_sec_zh,    "dist_secondary_road")
r_dist_settle_sc_zh      <- scale_raster_zh(r_dist_settle_zh, "dist_settlement")
r_dist_rail_sc_zh        <- scale_raster_zh(r_dist_rail_zh,   "dist_railway")

# Linear predictor using the Aargau coefficients.
eta_zh <- (r_forest_dens_sc_zh      * get_beta("forest_density_sc"))      +
          (r_dist_forest_edge_sc_zh * get_beta("dist_forest_edge_sc"))    +
          (r_dist_water_sc_zh       * get_beta("dist_water_sc"))          +
          (r_dist_fenced_sc_zh      * get_beta("dist_fenced_road_sc"))    +
          (r_dist_main_sc_zh        * get_beta("dist_main_road_sc"))      +
          (r_dist_sec_sc_zh         * get_beta("dist_secondary_road_sc")) +
          (r_dist_settle_sc_zh      * get_beta("dist_settlement_sc"))     +
          (r_dist_rail_sc_zh        * get_beta("dist_railway_sc"))

# Normalise with Aargau's eta range for a seamless cross-canton surface.
S_zh <- (eta_zh - eta_min) / (eta_max - eta_min)
S_zh <- clamp(S_zh, lower = 0, upper = 1)

# Keeley transformation.
R_zh <- exp(KEELEY_C * (1 - S_zh))

# Burn-in. Same order as Aargau, plus the Zurich-specific ASP fences.
R_zh[!is.na(r_water_bin_zh) & r_water_bin_zh == 1 &
     is.na(r_large_lakes_zh)]                                <- R_max * 0.9
R_zh[r_infra_zh == 1]                                        <- R_max
R_zh[r_highway_zh == 1]                                      <- NA
R_zh[!is.na(r_large_lakes_zh) & r_large_lakes_zh == 1]       <- NA
R_zh[!is.na(r_fences_zh)      & r_fences_zh      == 1]       <- NA
R_zh[r_passages_zh == 1]                                     <- R_min

# Mask to the exact canton polygon (drops the 2 km edge ring).
S_zh <- mask(S_zh, zurich_vect); names(S_zh) <- "habitat_suitability_zurich"
R_zh <- mask(R_zh, zurich_vect); names(R_zh) <- "resistance_zurich"

writeRaster(S_zh, file.path(OUT_DIR, "Habitat_Suitability_Wildboar_Zurich_25m.tif"),
            overwrite = TRUE, NAflag = -9999)
writeRaster(R_zh, file.path(OUT_DIR, "Resistance_Keeley_Wildboar_Zurich_25m.tif"),
            overwrite = TRUE, NAflag = -9999)

cat(sprintf("  Zurich resistance range (finite cells): [%.2f, %.2f]\n",
            global(R_zh, "min", na.rm = TRUE)[[1]],
            global(R_zh, "max", na.rm = TRUE)[[1]]))
cat(sprintf("  Impassable cells (NA): %d\n",
            global(is.na(R_zh), "sum", na.rm = TRUE)[[1]]))
cat("== Pipeline complete. ==\n")


## Step 8: Diagnostic Plots

Visual sanity checks: the iSSF coefficient forest plot, the
suitability and resistance side-by-side maps for both cantons, and a
resistance histogram for Aargau.


In [ ]:
cat("== 8. Diagnostic plots ==\n")

# iSSF coefficient forest plot (already saved, display inline).
print(p_coef)

# Suitability and resistance maps for Aargau.
png(file.path(OUT_DIR, "suitability_resistance_maps_Aargau.png"),
    width = 2400, height = 1100, res = 200)
par(mfrow = c(1, 2))
plot(S, main = "Habitat Suitability (0 to 1)\nHigh = preferred during transit",
     col = hcl.colors(50, "Greens 3"), axes = FALSE)
plot(R, main = "Resistance (Keeley C = 4)\nLow = permeable, NA = impassable",
     col = rev(hcl.colors(50, "Inferno")), axes = FALSE)
dev.off()

# Suitability and resistance maps for Zurich.
png(file.path(OUT_DIR, "suitability_resistance_maps_Zurich.png"),
    width = 2400, height = 1100, res = 200)
par(mfrow = c(1, 2))
plot(S_zh, main = "Habitat Suitability Zurich (0 to 1)",
     col = hcl.colors(50, "Greens 3"), axes = FALSE)
plot(R_zh, main = "Resistance Zurich (Keeley C = 4)",
     col = rev(hcl.colors(50, "Inferno")), axes = FALSE)
dev.off()

# Aargau resistance histogram.
png(file.path(OUT_DIR, "resistance_histogram.png"),
    width = 1400, height = 800, res = 200)
r_vals <- na.omit(as.numeric(values(R)))
hist(r_vals, breaks = 60, col = "#D85A30", border = "white",
     xlab = "Resistance value", ylab = "Pixel count",
     main = "Distribution of Resistance Values: Aargau")
abline(v = R_max, col = "black", lty = 2, lwd = 1.5)
text(R_max * 0.96, par("usr")[4] * 0.88,
     "Absolute barrier\n(R_max)", adj = 1, cex = 0.7)
dev.off()

cat("== All diagnostic plots written to data/processed/. ==\n")
